In [ ]:
import numpy as np

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from dataset_loaders import *

In [ ]:
dataset_loader = load_harmdirect_dataset

In [ ]:
B_ALL_KEY = "b_all"
B_CAT_KEY = "b_cat"
H_ALL_KEY = "h_all"
H_CAT_KEY = "h_cat"

samples, _ = dataset_loader(None, B_ALL_KEY, B_CAT_KEY, H_ALL_KEY, H_CAT_KEY)

all_benign_samples = samples[B_ALL_KEY]
all_harmful_samples = samples[H_ALL_KEY]

In [ ]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

benign_emb = embed_model.encode(
    all_benign_samples,
    batch_size=32,
    convert_to_numpy=True,
    show_progress_bar=True
)

harmful_emb = embed_model.encode(
    all_harmful_samples,
    batch_size=32,
    convert_to_numpy=True,
    show_progress_bar=True
)

In [ ]:
# Within-Class Semantic Spread
def class_spread(embeddings):
    centroid = embeddings.mean(axis=0)
    distances = np.linalg.norm(embeddings - centroid, axis=1)
    return {
        "mean_distance": distances.mean(),
        "std_distance": distances.std()
    }, centroid

benign_spread, benign_centroid = class_spread(benign_emb)
harmful_spread, harmful_centroid = class_spread(harmful_emb)

# How internally coherent the two sets are
print(f"Benign mean distance: {float(benign_spread["mean_distance"])}, standard distance: {float(benign_spread["std_distance"])}")
print(f"Harmful mean distance: {float(harmful_spread["mean_distance"])}, standard distance: {float(harmful_spread["std_distance"])}")

In [ ]:
# Between-Class Separation
centroid_l2 = np.linalg.norm(benign_centroid - harmful_centroid)
print("Centroid L2 distance:", centroid_l2)

centroid_cos = cosine_similarity(
    benign_centroid.reshape(1, -1),
    harmful_centroid.reshape(1, -1)
)[0][0]

# Lower cosine means more semantic separation
print("Centroid cosine similarity:", centroid_cos)

In [ ]:
# Signal-to-Noise Ratio
# Tells you if separation is large relative to internal spread
avg_within_spread = (
    benign_spread["mean_distance"] + harmful_spread["mean_distance"]
) / 2

separation_ratio = centroid_l2 / avg_within_spread

# < 1 -> classes overlap heavily, = 1 -> moderate separation, > 1 strong to very distinct distributions
print("Separation / Within Spread Ratio:", separation_ratio)

In [ ]:
# Cross-Class Pairwise Similarity
# Differing values mean good separation, similar values indicate heavy overlap
cross_sim = cosine_similarity(benign_emb, harmful_emb)
mean_cross_cos = cross_sim.mean()

print("Mean cross-class cosine similarity:", mean_cross_cos)

within_benign = cosine_similarity(benign_emb).mean()
within_harmful = cosine_similarity(harmful_emb).mean()

print("Mean benign internal cosine:", within_benign)
print("Mean harmful internal cosine:", within_harmful)